In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_cheng22_astro_v2.h5ad'
adata = sc.read(f)
adata

AnnData object with n_obs × n_vars = 7697 × 15573
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'Time', 'Light', 'curated_cluster', 'pc1', 'pc2'
    var: 'feature_types'
    uns: 'leiden', 'ngbr_astro'
    obsm: 'pc_astro'
    layers: 'norm'
    obsp: 'ngbr_astro_connectivities', 'ngbr_astro_distances'

In [3]:
adata.obs['Type']

AAAGGGCCATCGAAGG-1-P28_1a-1       Astro_A
AACAACCTCTAACGCA-1-P28_1a-1       Astro_A
AACAAGATCTAATTCC-1-P28_1a-1       Astro_A
AACACACCAGACCTAT-1-P28_1a-1       Astro_A
AACAGGGGTTGTGGCC-1-P28_1a-1       Astro_A
                                   ...   
ATCGATGCATGCCATA-1-P38_dr_2b-5    Astro_A
TCTACCGCATCGGCCA-1-P38_dr_2b-5    Astro_A
GGGAAGTTCCATGATG-1-P38_dr_2b-5    Astro_A
AAGCGTTGTGTTGACT-1-P38_dr_2b-5    Astro_A
TTGTGGAAGCTCTATG-1-P38_dr_2b-5    Astro_A
Name: Type, Length: 7697, dtype: category
Categories (2, object): ['Astro_A', 'Astro_B']

In [4]:
%%time

obs_fixed1 = 'Time'
obs_fixed2 = 'Light'
obs_random = 'Sample'

offset = 1e-2
scale = 1e4

for cluster in ['Astro_A', 'Astro_B']:
    tag = f'd260210_{cluster}'
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_astro_cheng22_{tag}.h5ad')


    adatasub = adata[adata.obs['Type']==cluster]

    # ### test
    # adatasub = adatasub[:,:20]
    # ### test

    genes = adatasub.var.index.values 

    obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()

    adatasub = adatasub[obs.index]

    # mat (CP10k norm)
    mat = np.array(adatasub.X.todense())/adatasub.obs['n_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm_two_fixed(mat, genes, obs, obs_fixed1, obs_fixed2, obs_random, output=output, offset=offset)
    print(output)

(6649, 15573) (6649, 3)
(6649, 15573) (6649, 3)
(6649, 10965) (6649, 3)


100% 10965/10965 [32:46<00:00,  5.57it/s] 


/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_astro_cheng22_d260210_Astro_A.h5ad
(1048, 15573) (1048, 3)
(1048, 15375) (1048, 3)
(1048, 13534) (1048, 3)


100% 13534/13534 [19:39<00:00, 11.47it/s]


/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_astro_cheng22_d260210_Astro_B.h5ad
CPU times: user 52min 21s, sys: 11 s, total: 52min 32s
Wall time: 52min 36s
